<a href="https://colab.research.google.com/github/Fayeyan/machine-learning-zoomcamp/blob/master/Homework/Homework_3_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [2]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'

In [3]:
!wget $data -O hw-week-3.csv

--2025-10-14 04:39:33--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘hw-week-3.csv’

hw-week-3.csv       100%[===================>]  78.98K  --.-KB/s    in 0.01s   

2025-10-14 04:39:33 (6.21 MB/s) - ‘hw-week-3.csv’ saved [80876/80876]



In [4]:
df = pd.read_csv('hw-week-3.csv')
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1




## Data preparation

* Check if the missing values are presented in the features
* If there are missing values:
  *  For categorical features, repalce them with 'NA'
  *  For numerical features, replace with 0.0



In [5]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

target_column = 'converted'
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)
numerical_colmns = (df.dtypes[df.dtypes != 'object'].index.difference([target_column]).to_list())

In [6]:
categorical_columns

['lead_source', 'industry', 'employment_status', 'location']

In [7]:
numerical_colmns

['annual_income',
 'interaction_count',
 'lead_score',
 'number_of_courses_viewed']

In [8]:
for c in categorical_columns:
  df[c] = df[c].fillna("NA")

for c in numerical_colmns:
  df[c] = df[c].fillna(0.0)

df.isnull().sum()

,0
lead_source,0
industry,0
number_of_courses_viewed,0
annual_income,0
employment_status,0
location,0
interaction_count,0
lead_score,0
converted,0


 Q1 What is the most frequent observation (mode) for the column industry?

In [9]:
df.industry.value_counts()

,count
industry,
retail,203
finance,200
other,198
healthcare,187
education,187
technology,179
manufacturing,174
NA,134


Q2 Correlation matrix of the numerical features of the dataset.

In [10]:
df[numerical_colmns].corr()

,annual_income,interaction_count,lead_score,number_of_courses_viewed
annual_income,1.000000,0.027036,0.015610,0.009770
interaction_count,0.027036,1.000000,0.009888,-0.023565
lead_score,0.015610,0.009888,1.000000,-0.004879
number_of_courses_viewed,0.009770,-0.023565,-0.004879,1.000000


## Split the data
* Split the data in train/val/test sets with 60%/20%/20%
* Use Scikit-Learn for that and set the seed to 42
* Make sure that the target value converted is not in dataframe.

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [13]:
len(df_train), len(df_val), len(df_test)

(876, 293, 293)

In [14]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [15]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

## Question 3
* Calculate mutual score between converted and other categorical variables in the dataset. Use the training set only.
* Round the scores to 2 decimals using round(scores, 2).

In [16]:
from sklearn.metrics import mutual_info_score

In [17]:
def mutual_info_converted_score(series):
  return round(mutual_info_score(series, df_train.converted), 2)

In [18]:

mi = df_train[categorical_columns].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

,0
lead_source,0.04
industry,0.01
employment_status,0.01
location,0.00


##Question 4
* Train a logistic regression.
* One-hot encoding for categorical variables.
* Fit on training dataset.

In [19]:
from sklearn.feature_extraction import DictVectorizer

In [57]:
dv = DictVectorizer(sparse=True)

train_dict = df_train[categorical_columns + numerical_colmns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_colmns].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [58]:
from sklearn.linear_model import LogisticRegression

In [59]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [60]:
model.intercept_[0]

np.float64(-0.0691472802783609)

In [61]:
model.coef_[0].round(3)

array([-0.   , -0.015,  0.034,  0.003,  0.012, -0.103, -0.025,  0.049,
       -0.02 , -0.013, -0.003, -0.009, -0.032, -0.016,  0.311,  0.051,
        0.02 , -0.012, -0.012, -0.115,  0.08 , -0.03 ,  0.004, -0.011,
       -0.011, -0.006,  0.008,  0.006, -0.033, -0.025,  0.454])

In [62]:
from sklearn.metrics import accuracy_score

In [63]:
acc_score = accuracy_score(y_val, model.predict(X_val))
acc_score

0.6996587030716723

In [64]:
round(acc_score, 2)

0.7

## Question 5
* Let's find the least useful feature using the feature elimination technique.
* Train a model using the same features and parameters as in Q4 without rounding.
* Exclude each feature from this set and train a model without it. Record the accuracy for each model.
* For each feature, calculate the difference between the origianl accurarcy and the accurarcy without the feature.
  

In [65]:
diff = {}
for feature in categorical_columns + numerical_colmns:
    features = categorical_columns + numerical_colmns
    features.remove(feature)
    dv = DictVectorizer(sparse=True)
    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    acc_score_new = accuracy_score(y_val, model.predict(X_val))
    diff[feature] = acc_score - acc_score_new

In [72]:
min(diff, key=lambda k: abs(diff[k]))

'industry'

## Question 6
* Train a regularized logistic regression.
* Let's try C: [0.01, 0.1, 1, 10, 100]
* Train models
* Calculate the accuracy on the val dataset and round it to 3 decimal digits.

In [78]:
dv = DictVectorizer(sparse=True)
train_dict = df_train[categorical_columns + numerical_colmns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)
val_dict = df_val[categorical_columns + numerical_colmns].to_dict(orient='records')
X_val = dv.transform(val_dict)
for c in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    acc_score = accuracy_score(y_val, model.predict(X_val))
    print(f'C: {c}, accuracy: {round(acc_score, 6)}')

C: 0.01, accuracy: 0.699659
C: 0.1, accuracy: 0.699659
C: 1, accuracy: 0.699659
C: 10, accuracy: 0.699659
C: 100, accuracy: 0.699659
